### 決定木とは
- 条件分岐を繰り返してデータを分類する手法
- 特徴量の閾値でデータを分割していく

||内容|
|----|----|
|モデル|条件分岐の木構造|
|分割基準|ジニ不純度の最小化（情報利得の最大化）|
|木の構築|再帰的な最良分割の探索|
|予測|ルートから葉まで条件を辿り、クラスを返す|
|過学習リスク|深さ無制限だと訓練データに過適合しやすい|


決定木は単体では過学習しやすいという弱点があります。これを解決するのがランダムフォレスト（複数の決定木のアンサンブル）

In [ ]:
# Irisデータセットの全3クラスを使用

import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(criterion="gini", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


木の成長
1. ルートノード(全サンプル)から始める
2. 情報利得が最大になる特徴量と閾値で分割する
3. 各子ノードに対して再帰的に同じ処理を繰り返す
4. 以下のいずれかで停止する
    - ノード内のサンプルが全て同じクラスになった
    - 最大の深さ（`max_depth`）に達した
    - ノード内のサンプル数が最小値を下回った

In [ ]:
# scratchで決定木を実装

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature   = feature    # 分割に使う特徴量のインデックス
        self.threshold = threshold  # 分割の閾値
        self.left      = left       # 左の子ノード（条件を満たす）
        self.right     = right      # 右の子ノード（条件を満たさない）
        self.value     = value      # 葉ノードの場合、予測クラス


class DecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _gini(self, y):
        n = len(y)
        if n == 0:
            return 0
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / n
        return 1 - np.sum(probs ** 2)

    def _information_gain(self, y, y_left, y_right):
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)
        return self._gini(y) - (n_l / n) * self._gini(y_left) - (n_r / n) * self._gini(y_right)

    def _best_split(self, X, y):
        best_gain = -1
        best_feature, best_threshold = None, None

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask  = X[:, feature] <= threshold
                right_mask = ~left_mask

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                gain = self._information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth=0):
        n_samples = len(y)
        n_classes = len(np.unique(y))

        # 停止条件
        if (n_classes == 1 or
            n_samples < self.min_samples_split or
            (self.max_depth is not None and depth >= self.max_depth)):
            leaf_value = np.bincount(y).argmax()  # 最頻クラスを予測値にする
            return Node(value=leaf_value)

        feature, threshold = self._best_split(X, y)
        if feature is None:
            leaf_value = np.bincount(y).argmax()
            return Node(value=leaf_value)

        left_mask  = X[:, feature] <= threshold
        right_mask = ~left_mask

        left  = self._build_tree(X[left_mask],  y[left_mask],  depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=feature, threshold=threshold, left=left, right=right)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


# 実行
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=None)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000
